In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/media_recommender/data', exist_ok=True)
!cp -r /content/drive/MyDrive/media_recommender_data/* /content/media_recommender/data/

import pickle
import numpy as np
import json
import pandas as pd
import shutil
import time
import torch
import torch.nn as nn
from sklearn.decomposition import TruncatedSVD
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
import psutil
from scipy.sparse import load_npz
import random
import gc

torch.manual_seed(42)

DATA_DIR = '/content/media_recommender/data'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install implicit

In [3]:
with open(os.path.join(DATA_DIR, "als_model.pkl"), "rb") as f:
    model = pickle.load(f)
with open(os.path.join(DATA_DIR, "anilist_to_mal.json"), "r") as f:
    mal_ids = json.load(f)
with open(os.path.join(DATA_DIR, "anime_data.jsonl"), "r") as f:
    anime_content = [json.loads(line) for line in f]

train = pd.read_parquet(os.path.join(DATA_DIR, "train_ratings.parquet"))
test = pd.read_parquet(os.path.join(DATA_DIR, "test_ratings.parquet"))

animes = pd.read_parquet(os.path.join(DATA_DIR, "animes.parquet"))

In [4]:
anime_tag_vectors = np.load(os.path.join(DATA_DIR, "anime_tag_vectors.npy"))
manga_tag_vectors = np.load(os.path.join(DATA_DIR, "manga_tag_vectors.npy"))

als_user_factors = model.user_factors
als_item_factors = model.item_factors

In [5]:
anime_ids = train['anime_id'].astype('category')
anime_id_map = dict(enumerate(anime_ids.cat.categories))
anime_id_map_reverse = {v: k for k, v in anime_id_map.items()}

user_ids = train['user_id'].astype('category')
user_id_map = dict(enumerate(user_ids.cat.categories))
user_id_map_reverse = {v: k for k, v in user_id_map.items()}
train['user_idx'] = user_ids.cat.codes

# AniList ID -> real MAL ID (via idMal crosswalk; drop failed lookups)
anilist_to_mal = {int(k): v for k, v in mal_ids.items() if v is not None}

none_count = sum(1 for v in mal_ids.values() if v is None)
print(f"anilist_to_mal: {none_count}/{len(mal_ids)} AniList entries had no MAL match (idMal was null) -- dropped")

# real MAL ID -> ratings dataset's internal animeID (bridge step that was missing before)
animes['mal_id'] = animes['mal_url'].str.extract(r'/anime/(\d+)').astype(int)

# --- sanity-check the bridge tables before trusting a dict built from them ---
dup_mal = animes['mal_id'].duplicated(keep=False)
dup_animeid = animes['animeID'].duplicated(keep=False)
if dup_mal.any():
    print(f"WARNING: {dup_mal.sum()} rows in animes share a duplicated mal_id -- "
          f"dict(zip(...)) will silently keep only the last row per key")
if dup_animeid.any():
    print(f"WARNING: {dup_animeid.sum()} rows in animes share a duplicated animeID")
if not dup_mal.any() and not dup_animeid.any():
    print("animes['mal_id'] and animes['animeID'] are both unique")

mal_to_animeid = dict(zip(animes['mal_id'], animes['animeID']))

# --- walk the full chain: AniList idx -> real MAL id -> dataset animeID -> ALS row ---
aligned_rows = []
for i, a in enumerate(anime_content):
    anilist_id = a['id']
    real_mal_id = anilist_to_mal.get(anilist_id)
    if real_mal_id is None:
        continue  # no MAL match for this AniList entry


    animeid = mal_to_animeid.get(real_mal_id)
    if animeid is None:
        continue  # MAL id doesn't appear in the ratings dataset at all

    als_row = anime_id_map_reverse.get(animeid)
    if als_row is None:
        continue  # in the ratings dataset, but never rated in `train` -> no ALS row

    aligned_rows.append({
        'anilist_idx': i,
        'real_mal_id': real_mal_id,
        'animeid': animeid,
        'als_row': als_row,
    })

aligned_df = pd.DataFrame(aligned_rows)
print(f"Aligned {len(aligned_df)} / {len(anime_content)} AniList anime through all three ID systems to an ALS row")


anilist_to_mal: 12/5000 AniList entries had no MAL match (idMal was null) -- dropped
animes['mal_id'] and animes['animeID'] are both unique
Aligned 4769 / 5000 AniList anime through all three ID systems to an ALS row


In [6]:
def anilist_title(a):
    # AniList title is a dict, not a plain string
    t = a['title']
    return t.get('english') or t.get('romaji') or t.get('native')

animes_title_by_id = animes.set_index('animeID')['title']

sample = aligned_df.sample(min(10, len(aligned_df)), random_state=2)
for _, row in sample.iterrows():
    anilist_t = anilist_title(anime_content[row['anilist_idx']])
    ratings_t = animes_title_by_id.loc[row['animeid']]
    print(f"AniList: {anilist_t!r:55} | animes: {ratings_t!r}")


AniList: 'The Seven Deadly Sins: Cursed by Light'                | animes: 'The Seven Deadly Sins the Movie 2: Cursed By Light'
AniList: 'Overlord: Ple Ple Pleiades 3'                          | animes: 'Overlord: Ple Ple Pleiades 3'
AniList: 'Tari Tari'                                             | animes: 'Tari Tari'
AniList: 'A Certain Magical Index Specials'                      | animes: 'A Certain Magical Index: Specials'
AniList: 'Legend of the Galactic Heroes'                         | animes: 'Legend of the Galactic Heroes'
AniList: 'Inazuma Eleven: The Seal of Orion'                     | animes: 'Inazuma Eleven: Orion no Kokuin'
AniList: 'Redline'                                               | animes: 'Redline'
AniList: 'The Helpful Fox Senko-san'                             | animes: 'The Helpful Fox Senko-san'
AniList: 'Komi Can’t Communicate'                                | animes: "Komi Can't Communicate"
AniList: 'Pale Cocoon'                                          

In [7]:
combined_item_features = []
for _, row in aligned_df.iterrows():
    als_vec = als_item_factors[row['als_row']]
    tag_vec = anime_tag_vectors[row['anilist_idx']]
    combined = np.concatenate([als_vec, tag_vec])
    combined_item_features.append(combined)

combined_item_features = np.array(combined_item_features)

aligned_animeids = set(aligned_df['animeid'])

train_aligned = train[train['anime_id'].isin(aligned_animeids)]

print(f"Original train rows: {len(train):,}")
print(f"Filtered to aligned anime: {len(train_aligned):,}")
print(f"Unique users remaining: {train_aligned['user_id'].nunique():,}")

del train
gc.collect()

Original train rows: 118,387,327
Filtered to aligned anime: 112,808,506
Unique users remaining: 1,773,743


13

In [8]:
positive_counts = train_aligned[train_aligned['is_positive'] == 1].groupby('user_id').size()
qualifying_users = positive_counts[positive_counts >= 5].index
train_final = train_aligned[train_aligned['user_id'].isin(qualifying_users)]

del train_aligned, positive_counts, qualifying_users
gc.collect()

7

In [9]:
MAX_POSITIVES_PER_USER = 50

train_positive = train_final[train_final['is_positive'] == 1][['user_id', 'anime_id']].copy()

# Assign a random number per row, sort by user + random, then rank within each user group
train_positive['rand'] = np.random.rand(len(train_positive))
train_positive = train_positive.sort_values(['user_id', 'rand'])
train_positive['rank'] = train_positive.groupby('user_id').cumcount()

train_capped = train_positive[train_positive['rank'] < MAX_POSITIVES_PER_USER][['user_id', 'anime_id']]
del train_positive

print(f"Training pairs after capping: {len(train_capped):,}")

Training pairs after capping: 39,185,238


In [10]:
animeid_to_aligned_idx = {row['animeid']: i for i, row in aligned_df.reset_index(drop=True).iterrows()}
user_id_to_useridx = train_final[['user_id', 'user_idx']].drop_duplicates().set_index('user_id')['user_idx'].to_dict()

In [11]:
class TwoTowerDataset(Dataset):
    def __init__(self, train_capped, animeid_to_aligned_idx, user_id_to_useridx, aligned_animeids):
        # Only keep rows where the anime is actually in our aligned set
        # (should already be true given train_final's filtering, but worth being defensive)
        self.data = train_capped.reset_index(drop=True)
        self.animeid_to_aligned_idx = animeid_to_aligned_idx
        self.user_id_to_useridx = user_id_to_useridx
        self.aligned_animeids = list(aligned_animeids)  # for fast random.choice sampling

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        user_idx = self.user_id_to_useridx[row['user_id']]
        pos_anime_idx = self.animeid_to_aligned_idx[row['anime_id']]

        # Sample a negative anime at random from the aligned set
        neg_anime_id = random.choice(self.aligned_animeids)
        neg_anime_idx = self.animeid_to_aligned_idx[neg_anime_id]

        return {
            'user_idx': user_idx,
            'pos_item_idx': pos_anime_idx,
            'neg_item_idx': neg_anime_idx,
        }

In [12]:
aligned_animeids = set(aligned_df['animeid'])

dataset = TwoTowerDataset(train_capped, animeid_to_aligned_idx, user_id_to_useridx, aligned_animeids)

BATCH_SIZE = 16384

dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

# Pull one batch to confirm the shapes look right
batch = next(iter(dataloader))

In [13]:
class UserTower(nn.Module):
    def __init__(self, input_dim=64, output_dim=64, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.net(x)


class ItemTower(nn.Module):
    def __init__(self, input_dim=486, output_dim=64, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.net(x)

In [14]:
item_input_dim = combined_item_features.shape[1]

user_tower = UserTower()
item_tower = ItemTower(input_dim=item_input_dim)

optimizer = optim.Adam(
    list(user_tower.parameters()) + list(item_tower.parameters()),
    lr=0.001
)

def bpr_loss(user_vec, pos_vec, neg_vec):
    pos_score = (user_vec * pos_vec).sum(dim=1)
    neg_score = (user_vec * neg_vec).sum(dim=1)
    return -torch.log(torch.sigmoid(pos_score - neg_score) + 1e-8).mean()

In [15]:
CHECKPOINT_PATH = os.path.join(DATA_DIR, "two_tower_checkpoint.pt")

if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH)
    user_tower = UserTower()
    item_tower = ItemTower(input_dim=item_input_dim)
    user_tower.load_state_dict(checkpoint['user_tower_state'])
    item_tower.load_state_dict(checkpoint['item_tower_state'])

    optimizer = optim.Adam(
        list(user_tower.parameters()) + list(item_tower.parameters()),
        lr=0.001
    )
    optimizer.load_state_dict(checkpoint['optimizer_state'])

    start_epoch = checkpoint['epoch'] + 1
    print(f"Resuming from epoch {start_epoch + 1}")
else:
    user_tower = UserTower()
    item_tower = ItemTower(input_dim=item_input_dim)
    optimizer = optim.Adam(
        list(user_tower.parameters()) + list(item_tower.parameters()),
        lr=0.001
    )
    start_epoch = 0
    print("Starting fresh — no checkpoint found")

# --- Train ---
EPOCHS = 3  # the real total target, not just 1

for epoch in range(start_epoch, EPOCHS):
    total_loss = 0
    num_batches = 0

    for batch in dataloader:
        user_vecs = torch.tensor(als_user_factors[batch['user_idx']], dtype=torch.float32)
        pos_vecs = torch.tensor(combined_item_features[batch['pos_item_idx']], dtype=torch.float32)
        neg_vecs = torch.tensor(combined_item_features[batch['neg_item_idx']], dtype=torch.float32)

        user_out = user_tower(user_vecs)
        pos_out = item_tower(pos_vecs)
        neg_out = item_tower(neg_vecs)

        loss = bpr_loss(user_out, pos_out, neg_out)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

        if num_batches % 500 == 0:
            print(f"Epoch {epoch+1}, batch {num_batches}, avg loss: {total_loss/num_batches:.4f}")

    epoch_loss = total_loss / num_batches
    print(f"Epoch {epoch+1} complete — avg loss: {epoch_loss:.4f}")

        # --- Save after EVERY epoch ---
    torch.save({
        'epoch': epoch,
        'user_tower_state': user_tower.state_dict(),
        'item_tower_state': item_tower.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'loss': epoch_loss,
    }, CHECKPOINT_PATH)
    print(f"Checkpoint saved after epoch {epoch+1}")

    shutil.copytree('/content/media_recommender/data', '/content/drive/MyDrive/media_recommender_data', dirs_exist_ok=True)
    print(f"Backed up to Drive after epoch {epoch+1}")

Resuming from epoch 4


In [16]:
user_tower.eval()
item_tower.eval()

with torch.no_grad():
    all_item_vecs = torch.tensor(combined_item_features, dtype=torch.float32)
    all_item_embeddings = item_tower(all_item_vecs).numpy()

print(all_item_embeddings.shape)

(4769, 64)


In [17]:
def two_tower_recommend_for_user(user_id, k=10):
    if user_id not in user_id_to_useridx:
        return []

    user_idx = user_id_to_useridx[user_id]
    user_vec = torch.tensor(als_user_factors[user_idx], dtype=torch.float32).unsqueeze(0)

    with torch.no_grad():
        user_embedding = user_tower(user_vec).numpy()[0]

    scores = all_item_embeddings @ user_embedding  # dot product against every aligned anime

    # Exclude anime this user already positively rated
    already_rated = set(train_final[
        (train_final['user_id'] == user_id) & (train_final['is_positive'] == 1)
    ]['anime_id'])

    aligned_animeid_list = aligned_df['animeid'].tolist()
    ranked_indices = scores.argsort()[::-1]

    recommendations = []
    for idx in ranked_indices:
        animeid = aligned_animeid_list[idx]
        if animeid not in already_rated:
            recommendations.append(animeid)
        if len(recommendations) == k:
            break

    return recommendations

In [18]:
user_item_matrix_full = load_npz(os.path.join(DATA_DIR, "item_user_matrix.npz"))
user_item_matrix_full = user_item_matrix_full.T.tocsr()


def als_recommend_for_user(user_idx, k=10):
    recommended = model.recommend(
        user_idx,
        user_item_matrix_full[user_idx],
        N=k
    )
    item_indices, scores = recommended
    return [anime_id_map[i] for i in item_indices]

In [19]:
for _name in ['train_capped', 'dataset', 'dataloader', 'batch']:
    if _name in globals():
        del globals()[_name]
train_final = train_final[['user_id', 'anime_id', 'is_positive', 'user_idx']]
gc.collect()


aligned_animeid_list = aligned_df['animeid'].tolist()

def precision_recall_two_tower(test_df, k=10, sample_users=2000):
    aligned_animeids = set(aligned_df['animeid'])

    # Slim columns first, filter to aligned positives
    test_pos = test_df.loc[test_df['is_positive'] == 1, ['user_id', 'anime_id']]
    test_aligned = test_pos[test_pos['anime_id'].isin(aligned_animeids)]
    del test_pos

    # Unique users WITHOUT groupby.groups — vectorized, cheap
    eligible = test_aligned['user_id'].unique()
    users_to_eval = [u for u in eligible if u in user_id_to_useridx]
    if sample_users and len(users_to_eval) > sample_users:
        users_to_eval = list(np.random.choice(users_to_eval, size=sample_users, replace=False))
    eval_user_set = set(users_to_eval)

    # NOW shrink to sampled users, THEN group — groupby over ~2000 users' rows, not 1.7M
    test_eval = test_aligned[test_aligned['user_id'].isin(eval_user_set)]
    del test_aligned
    actual_positive_lookup = test_eval.groupby('user_id')['anime_id'].agg(set).to_dict()
    del test_eval

    # Already-rated lookup from train, scoped to sampled users (as before)
    eval_positives = train_final[
        (train_final['is_positive'] == 1) & (train_final['user_id'].isin(eval_user_set))
    ][['user_id', 'anime_id']]
    user_positive_lookup = eval_positives.groupby('user_id')['anime_id'].agg(set).to_dict()
    del eval_positives
    gc.collect()

    user_indices = [user_id_to_useridx[u] for u in users_to_eval]
    user_vecs = torch.tensor(als_user_factors[user_indices], dtype=torch.float32)

    with torch.no_grad():
        user_embeddings = user_tower(user_vecs).numpy()

    all_scores = user_embeddings @ all_item_embeddings.T

    total_relevant = 0
    total_hits = 0
    start = time.time()

    for i, user_id in enumerate(users_to_eval):
        actual_positive = actual_positive_lookup[user_id]
        already_rated = user_positive_lookup.get(user_id, set())

        ranked_indices = all_scores[i].argsort()[::-1]

        recommended = []
        for idx in ranked_indices:
            animeid = aligned_animeid_list[idx]
            if animeid not in already_rated:
                recommended.append(animeid)
            if len(recommended) == k:
                break

        total_hits += len(actual_positive & set(recommended))
        total_relevant += len(actual_positive)


    recall = total_hits / total_relevant if total_relevant > 0 else 0
    precision = total_hits / (len(users_to_eval) * k)
    return precision, recall

In [20]:
precision, recall = precision_recall_two_tower(test, k=10, sample_users=2000)
print(f"Two-Tower — Precision@10: {precision:.4f}")
print(f"Two-Tower — Recall@10: {recall:.4f}")

Two-Tower — Precision@10: 0.2118
Two-Tower — Recall@10: 0.1822


In [ ]:
anime_ids = train['anime_id'].astype('category')
anime_id_map = dict(enumerate(anime_ids.cat.categories))
with open(os.path.join(DATA_DIR, 'anime_id_map.json'), 'w') as f:
    json.dump({int(k): int(v) for k, v in anime_id_map.items()}, f)

In [21]:
drive.mount('/content/drive')

shutil.copytree('/content/media_recommender/data', '/content/drive/MyDrive/media_recommender_data', dirs_exist_ok=True)
print(os.listdir('/content/drive/MyDrive/media_recommender_data'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['train_ratings.parquet', 'test_ratings.parquet', 'item_user_matrix.npz', 'item_similarity.npz', 'als_model.pkl', 'manga_tag_vectors.npy', 'anime_tag_vectors.npy', 'animes.parquet', 'animes.csv.parquet', 'manga_ratings.parquet', 'manga_data.jsonl', 'anime_data.jsonl', 'anilist_to_mal.json', 'two_tower_checkpoint.pt']
